<a href="https://colab.research.google.com/github/rida09Heythere/wellness-anxiety-insight-assistant/blob/main/data_cleaning_newV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gspread oauth2client

In [49]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials
scope=["https://spreadsheets.google.com/feeds",
       "https://www.googleapis.com/auth/drive"]
creds=ServiceAccountCredentials.from_json_keyfile_name("cycle-tracker-r.json",scope)
client=gspread.authorize(creds)

sheet = client.open("Cycle-Tracker-data").worksheet("Health Responses")
print("connected successfully")

connected successfully


In [50]:
import pandas as pd
print(sheet.get_all_records)
df=pd.DataFrame(sheet.get_all_records())
print(df.columns)


<bound method Worksheet.get_all_records of <Worksheet 'Health Responses' id:2089203399>>
Index(['Date', 'Gender', 'Name', 'Cycle Phase', 'Sleep Hrs/Day',
       'Exercise Frequency', 'Anxiety Score (1-10)', 'Anxiety Triggers',
       'Coping Methods', 'Suggestion Helped', 'Most Anxious Phase',
       'Symptoms in during anxious phase'],
      dtype='object')


In [51]:
df.shape
df.info()
df.isnull()
df["Gender"].value_counts()
df.duplicated().sum()
print(df.columns.tolist())
print(df["Gender"].value_counts())
print(df["Cycle Phase"].unique())
print(df["Cycle Phase"].unique())
print(df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 12 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   Date                              20 non-null     object
 1   Gender                            20 non-null     object
 2   Name                              20 non-null     object
 3   Cycle Phase                       20 non-null     object
 4   Sleep Hrs/Day                     20 non-null     int64 
 5   Exercise Frequency                20 non-null     object
 6   Anxiety Score (1-10)              20 non-null     int64 
 7   Anxiety Triggers                  20 non-null     object
 8   Coping Methods                    20 non-null     object
 9   Suggestion Helped                 20 non-null     object
 10  Most Anxious Phase                20 non-null     object
 11  Symptoms in during anxious phase  20 non-null     object
dtypes: int64(2), object(10)


In [52]:
df.groupby("Gender")["Anxiety Score (1-10)"].mean()



,Anxiety Score (1-10)
Gender,
Female,6.090909
Male,5.444444


In [53]:
df.groupby("Anxiety Triggers")["Anxiety Score (1-10)"].mean().sort_values(ascending=False)



,Anxiety Score (1-10)
Anxiety Triggers,
"Financial , Family",8.000000
"Family,Future,Relationship,",7.000000
Family,7.000000
Responsibilities,7.000000
Health,7.000000
Career,6.750000
"Relationship, Career",6.000000
"Financial, Health, Relationship, Low self-esteem, Future, Family, Studies, Career",6.000000
Financial,5.666667


In [54]:
female_df=df[df["Gender"]=="Female"]
female_df.groupby("Cycle Phase")["Anxiety Score (1-10)"].mean()



,Anxiety Score (1-10)
Cycle Phase,
Follicular,4.000000
Luteal,5.500000
Menstrual,5.800000
Ovulation,7.666667


In [55]:
male_df=df[df["Gender"]=="Male"]
male_df.groupby("Cycle Phase")["Anxiety Score (1-10)"].mean()



,Anxiety Score (1-10)
Cycle Phase,
N/A,5.444444


In [56]:
df["Sleep Hrs/Day"].corr(df["Anxiety Score (1-10)"])



np.float64(-0.3787921467962709)

In [57]:
df.groupby("Exercise Frequency")["Anxiety Score (1-10)"].mean()



,Anxiety Score (1-10)
Exercise Frequency,
1-2 days a week,5.750000
3-5 days a week,6.000000
Daily,4.333333
Never,6.222222


In [58]:
df["Anxiety Triggers"].value_counts()

,count
Anxiety Triggers,
Career,4
Financial,3
Relationship,2
"Family,Future,Relationship,",2
"Financial, Health, Relationship, Low self-esteem, Future, Family, Studies, Career",1
Health,1
"Relationship, Career",1
Responsibilities,1
none,1


In [59]:
from collections import Counter
all_methods=[]
for methods in df["Coping Methods"]:all_methods.extend([m.strip() for m in methods.split(",")])
Counter(all_methods).most_common()



[('Eating', 7),
 ('Music', 5),
 ('Sleeping', 5),
 ('Droomscroll/netflix', 4),
 ('Prayer', 3),
 ('Exercise', 2),
 ('Talking to a friend/family', 1),
 ('Self harm', 1),
 ('none', 1),
 ('Travelling', 1),
 ('Talking to friend/family', 1)]

In [60]:
df["Anxiety Score (1-10)"].describe()

,Anxiety Score (1-10)
count,20.000000
mean,5.800000
std,2.764436
min,1.000000
25%,4.000000
50%,6.500000
75%,7.000000
max,10.000000


In [61]:
from collections import Counter
all_triggers=[]
for triggers in df["Anxiety Triggers"]:all_triggers.extend([t.strip() for t in triggers.split(",")])
triggers_counts=Counter(all_triggers)
trigger_counts=pd.Series(Counter(all_triggers)).sort_values(ascending=False)
print(trigger_counts)

Relationship        7
Family              6
Career              6
Financial           5
Future              3
Health              3
                    2
Studies             2
Low self-esteem     1
Responsibilities    1
none                1
Social              1
dtype: int64


In [24]:
type(all_methods)

list

In [62]:
cleaned_methods=[m.strip().replace("Sleep","Sleeping").replace("Prayers","Prayer").replace("Talking to friend /family","Taking to friend/family").replace("Watching phone/movies etc","Drromscroll/netflix")
for m in all_methods]

from collections import Counter
Counter(cleaned_methods).most_common()
sorted(set(all_methods))


['Droomscroll/netflix',
 'Eating',
 'Exercise',
 'Music',
 'Prayer',
 'Self harm',
 'Sleeping',
 'Talking to a friend/family',
 'Talking to friend/family',
 'Travelling',
 'none']

In [63]:
from collections import Counter
all_coping=[]
for coping in df["Coping Methods"]:all_coping.extend([t.strip() for t in coping.split(",")])
coping_counts=Counter(all_coping)
copings_counts=pd.Series(Counter(all_coping)).sort_values(ascending=False)
print(copings_counts)

Eating                        7
Music                         5
Sleeping                      5
Droomscroll/netflix           4
Prayer                        3
Exercise                      2
Talking to a friend/family    1
Self harm                     1
none                          1
Travelling                    1
Talking to friend/family      1
dtype: int64
